In [1]:
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
import json
import pandas as pd

In [2]:

load_dotenv()


True

In [3]:
# import psycopg2

# conn = psycopg2.connect(
#     host=os.getenv("NEON_DB_HOST"),
#     database=os.getenv("NEON_DB_NAME"),
#     user=os.getenv("NEON_DB_USER_NAME"),
#     password=os.getenv("NEON_PASSWORD"),
#     port= 5432,
#     sslmode= "require"
# )


# cur = conn.cursor()

# cur.execute("SELECT version();")

# result = cur.fetchone()

# print(result)

# cur.close()
# conn.close()

DATABASE_URL = (
    f"postgresql://{os.getenv("NEON_DB_USER_NAME")}:{os.getenv("NEON_PASSWORD")}@{os.getenv("NEON_DB_HOST")}/{os.getenv("NEON_DB_NAME")}"
)

engine = create_engine(
    DATABASE_URL
)


In [4]:

def load_csv(path):

    df = pd.read_csv(path)

    # standardize columns

    df.columns = [
        c.strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")

        for c in df.columns
    ]

    return df


In [5]:
def convert_to_mm(unit, value):
    unit = unit.strip().upper()

    conversion = {
        "MM": 1,
        "CM": 10,
        "M": 1000,
        "IN": 25.4,
        "FT": 304.8
    }

    if unit not in conversion:
        unit = 'IN'
        # raise ValueError(f"Unsupported dimension unit: {unit}")

    return value * conversion[unit]


def convert_to_kg(unit, value):
    unit = unit.strip().upper()

    conversion = {
        "KG": 1,
        "G": 0.001,
        "MG": 0.000001,
        "LB": 0.453592,
        "OZ": 0.0283495
    }

    if unit not in conversion:
        unit = 'KG' 
        # raise ValueError(f"Unsupported weight unit: {unit}")

    return value * conversion[unit]



In [6]:
import pandas as pd


def safe_float(value):

    if pd.isna(value):
        return 0.0

    value = str(value)
    value = value.replace(",", "")
    value = value.strip()
    if value == "":
        return 0.0
    return float(value)

In [14]:
lane_transmode_df = load_csv(r'd:\acies_solutions\references\reference_data_in_csv\lane transmode route.csv')
route_master_df = load_csv(r'd:\acies_solutions\references\reference_data_in_csv\route master.csv')
shipping_detail_df = load_csv(r'd:\acies_solutions\references\reference_data_in_csv\shipment details.csv')
shipment_priority_df = load_csv(r'd:\acies_solutions\references\reference_data_in_csv\shipment priority.csv')
tlb_output_detauls_df = load_csv(r'd:\acies_solutions\references\reference_data_in_csv\tlb output details.csv')
tlb_output_header_df = load_csv(r'd:\acies_solutions\references\reference_data_in_csv\tlb output header.csv')
item_master_df = load_csv(r'd:\acies_solutions\references\reference_data_in_csv\item master.csv')


In [8]:
import re


def extract_int(value):
    if value is None:
        return -1
    value = str(value)
    match = re.search(r"-?\d+", value)
    if match:
        return int(match.group())
    return -1

In [9]:
# ==========================================
# CREATE TARGET DATAFRAME
# ==========================================

item_master_target_df = pd.DataFrame()

# ==========================================
# REQUIRED FIELDS
# ==========================================

item_master_target_df["sku_id"] = (
	item_master_df["sku_id"]
	.fillna(-1)
	.apply(extract_int)
	.astype("int64")
)


item_master_target_df["sku_name"] = (
	item_master_df["item_name"]
	.fillna("UNKNOWN")
)


item_master_target_df["length_mm"] = (
	item_master_df["length_mm"]
	.fillna(0)
)


item_master_target_df["width_mm"] = (
	item_master_df["width_mm"]
	.fillna(0)
)


item_master_target_df["height_mm"] = (
	item_master_df["height_mm"]
	.fillna(0)
)


item_master_target_df["weight_kg"] = (
    item_master_df["weight_kg"]
	.fillna(0)
    )


# ==========================================
# OPTIMIZER FIELDS
# ==========================================

item_master_target_df["stacking_limit"] = 3
item_master_target_df["can_rotate"] = True


# ==========================================
# TEMPERATURE
# ==========================================

item_master_target_df["temperature_min_c"] = 20
item_master_target_df["temperature_max_c"] = 60


# ==========================================
# HAZMAT / FRAGILITY
# ==========================================

item_master_target_df["hazmat_class"] = 0
item_master_target_df["fragility_rating"] = 1


# ==========================================
# SHELF LIFE
# ==========================================

item_master_target_df["shelf_life_days"] = 90


# ==========================================
# FOOD / REGULATED
# ==========================================

item_master_target_df["is_food_grade"] = False
item_master_target_df["is_regulated"] = False


# ==========================================
# REMOVE DUPLICATES
# ==========================================

item_master_target_df = (
	item_master_target_df
	.drop_duplicates(
		subset=["sku_id"]
	)
	.reset_index(drop=True)
)


# ==========================================
# PREVIEW
# ==========================================

item_master_target_df.head()


# # ==========================================
# # LOAD INTO POSTGRES
# # ==========================================

# item_master_target_df.to_sql(
#     name="item_master",
#     con=engine,
#     schema="public",
#     if_exists="append",
#     index=False,
#     method="multi",
#     chunksize=1000
# )

# print(
#     "item_master loaded successfully"
# )

,sku_id,sku_name,length_mm,width_mm,height_mm,weight_kg,stacking_limit,can_rotate,temperature_min_c,temperature_max_c,hazmat_class,fragility_rating,shelf_life_days,is_food_grade,is_regulated
0,122478,122478 - SUSTAIN JR CHOCO 500G SACH?PC,1000.00,1200.00,1600.000,658.0,3,True,20,60,0,1,90,False,False
1,208028,208028 - OIKOS DRINK HP MAN- PINEAPPLE 190MLX1(8),1219.20,1016.00,1621.028,653065.0,3,True,20,60,0,1,90,False,False
2,188275,188275 - PRO-STAT PEACH 96X1FL OZ,1070.00,1220.00,912000.000,556924.0,3,True,20,60,0,1,90,False,False
3,202849,202849 - Activia Kefir Strawberry 950ML,1191.26,967.74,1103.630,87245.0,3,True,20,60,0,1,90,False,False
4,206410,206410 - TWO GOOD STRAWBERRY 95GX4(6),1219.00,1016.00,150.000,63232.0,3,True,20,60,0,1,90,False,False


In [11]:
item_master_df.head()

,sku_id,item_name,length_mm,width_mm,height_mm,weight_kg,item.[item_base_uom],uom.[uom],weight_uom,weight,dimension_uom,length,width,height,height.1,floor_area,unit_count_in_pallet
0,122478,122478 - SUSTAIN JR CHOCO 500G SACH?PC,1000.00,1200.00,1600.000,658.0,PC,PAL - Pallet,KG,658,M,1,1.2,1.6,1,548,1248
1,208028,208028 - OIKOS DRINK HP MAN- PINEAPPLE 190MLX1(8),1219.20,1016.00,1621.028,653065.0,PC,PAL - Pallet,KG,653065,IN,48,40,63.82,"1,920",340,3960
2,188275,188275 - PRO-STAT PEACH 96X1FL OZ,1070.00,1220.00,912000.000,556924.0,PC,PAL - Pallet,KG,556924,M,1.07,1.22,912,1,"4,28,181",12672
3,202849,202849 - Activia Kefir Strawberry 950ML,1191.26,967.74,1103.630,87245.0,PC,PAL - Pallet,KG,"87,245",IN,46.9,38.1,43.45,"1,787",49,900
4,206410,206410 - TWO GOOD STRAWBERRY 95GX4(6),1219.00,1016.00,150.000,63232.0,PC,PAL - Pallet,KG,"63,232",MM,"1,219.00","1,016.00",150,"12,38,504",0,1536


In [10]:
def build_dimension_json(
    length,
    width,
    height
):
    return {
        "length_mm": int(
            safe_float(length)
        ),
        "width_mm": int(
            safe_float(width)
        ),
        "height_mm": int(
            safe_float(height)
        )
    }

In [ ]:
sku_uom_df = pd.DataFrame()

# ==========================================
# PRIMARY KEY
# ==========================================

sku_uom_df["sku_id"] = (
	item_master_df["sku_id"]
	.fillna(-1)
	.apply(extract_int)
	.astype("int64")
)

# ==========================================
# CASE DIMENSIONS
# ==========================================

sku_uom_df["case_dimensions"] = None

# ==========================================
# PALLET DIMENSIONS
# ==========================================

# Sample standard pallet sizing
# Replace later with actual data



sku_uom_df["pallet_dimensions"] =  (
    item_master_df.apply(
        lambda row:
        json.dumps(
            build_dimension_json(
                row["length_mm"],
                row["width_mm"],
                row["height_mm"]
            )
        ),
        axis=1
    )
)

# ==========================================
# BOX DIMENSIONS
# ==========================================

sku_uom_df["box_dimensions"] = None


# ==========================================
# UNIT COUNTS
# ==========================================

sku_uom_df["unit_count_in_case"] = 0

sku_uom_df["unit_count_in_pallet"] = (

    item_master_df[
        "unit_count_in_pallet"
    ]
    .fillna(1)
    .astype(int)
)

sku_uom_df["unit_count_in_box"] = 0


# ==========================================
# REMOVE DUPLICATES
# ==========================================

sku_uom_df = (
    sku_uom_df
    .drop_duplicates(
        subset=["sku_id"]
    )
    .reset_index(drop=True)
)


# ==========================================
# PREVIEW
# ==========================================

sku_uom_df.head()

### Write to SKU

# sku_uom_df.to_sql(
#     name="sku_unit_of_measure",
#     con=engine,
#     schema="public",
#     if_exists="append",
#     index=False,
#     method="multi",
#     chunksize=1000
# )

# print(
#     "sku_unit_of_measure loaded successfully"
# )



,sku_id,case_dimensions,pallet_dimensions,box_dimensions,unit_count_in_case,unit_count_in_pallet,unit_count_in_box
0,122478,None,"{""length_mm"": 1000, ""width_mm"": 1200, ""height_...",None,0,1248,0
1,208028,None,"{""length_mm"": 1219, ""width_mm"": 1016, ""height_...",None,0,3960,0
2,188275,None,"{""length_mm"": 1070, ""width_mm"": 1220, ""height_...",None,0,12672,0
3,202849,None,"{""length_mm"": 1191, ""width_mm"": 967, ""height_m...",None,0,900,0
4,206410,None,"{""length_mm"": 1219, ""width_mm"": 1016, ""height_...",None,0,1536,0


In [33]:
### Location

origin_locations = pd.DataFrame({
    "location_id":
        route_master_df[
            "origin_location_id"
        ],
    "location_name":
        route_master_df[
            "origin.location_name"
        ]
})

destination_locations = pd.DataFrame({

    "location_id":
        route_master_df[
            "destination_location_id"
        ],

    "location_name":
        route_master_df[
            "destination.location_name"
        ]
})


location_df = pd.concat([
    origin_locations,
    destination_locations
], ignore_index=True)

location_df["location_id"] = (
    location_df["location_id"]
    .astype(str)
    .str.strip()
)


location_df["location_name"] = (
    location_df["location_name"]
    .astype(str)
    .str.strip()
)


location_df = location_df[
    location_df["location_id"] != ""
]

location_df = location_df[
    location_df["location_id"] != "nan"
]

location_df["location_id"] = location_df["location_id"].apply(lambda x: str(x).zfill(4))

location_df = location_df.groupby(['location_id', 'location_name']).count().reset_index()


In [37]:
def infer_location_type(name):
    name = str(name).upper()
    if "WH" in name:
        return "WAREHOUSE"
    if "DC" in name:
        return "DISTRIBUTION_CENTER"
    if "STORE" in name:
        return "STORE"
    return "BUILDING"

In [ ]:
location_df["location_type"] = (
    location_df["location_name"]
    .apply(infer_location_type)
)
location_df["latitude"] = 0
location_df["longitude"] = 0
location_df["address"] = ""
location_df["city"] = ""
location_df["state"] = ""
location_df["country"] = ""
location_df["postal_code"] = ""
location_df["dock_count"] = 1
location_df["storage_type"] = ""
location_df["temperature_capability"] = False
location_df["operating_hours"] = ""

In [39]:
location_df['location_type'].value_counts()

location_type
DISTRIBUTION_CENTER    55
BUILDING               31
WAREHOUSE               1
Name: count, dtype: int64

In [40]:
# location_df.to_sql(
#     name="location",
#     con=engine,
#     schema="public",
#     if_exists="replace",
#     index=False,
#     method="multi",
#     chunksize=1000
# )

# print(
#     "location table loaded successfully"
# )

location table loaded successfully


In [42]:
route_master_df.head()

,item_name,origin.location_name,destination.location_name,origin_location_id,destination_location_id,lane_id_from_and_to,transport_asset_name,item_lane_transmode_association_final,item_lane_transmode_priority_final,item_lane_transmode_lead_time_final,cost_per_lb,min_lot_size.uom([uom].[uom].[pc]),item_quantity.uom([uom].[uom].[pc]),lane_type
0,101964 - LT N FIT GREEK TIRAMISU 5.3OZ,0156 - 0030 US Midwest DC-Minster,0155 - 0030 US PL MINSTER,156,155,Lane_From_0156_To_0155,TRUCK_CH_1DR,NaN,-1,NaN,NaN,NaN,NaN,Campus Lane
1,101964 - LT N FIT GREEK TIRAMISU 5.3OZ,0159 - 0030 US DC FW,0156 - 0030 US Midwest DC-Minster,159,156,Lane_From_0159_To_0156,TRUCK_CH_1DR,NaN,-1,NaN,NaN,NaN,NaN,External Input Lane
2,101964 - LT N FIT GREEK TIRAMISU 5.3OZ,0160 - 0030 US DC ALLENTOWN,0156 - 0030 US Midwest DC-Minster,160,156,Lane_From_0160_To_0156,TRUCK_CH_1DR,NaN,NaN,NaN,NaN,NaN,NaN,Planning Lane
3,101964 - LT N FIT GREEK TIRAMISU 5.3OZ,6015 - 0030 US DC Interchange,6006 - 0030 US DC Mt. Crawford,6015,6006,Lane_From_6015_To_6006,TRUCK_CH_1DR,NaN,-1,NaN,NaN,NaN,NaN,Campus Lane
4,101964 - LT N FIT GREEK TIRAMISU 5.3OZ,6037 - 0030 US DC US Cold Arlington,6010 - 0030 US DC Dallas,6037,6010,Lane_From_6037_To_6010,TRUCK_CH_1DR,NaN,-1,NaN,NaN,NaN,NaN,Campus Lane


In [52]:
lane_master_df = pd.DataFrame()
lane_master_df["lane_name"] = (

    route_master_df[
        "lane_id_from_and_to"
    ]
    .astype(str)
    .str.strip()
)

lane_master_df["lane_name"] = (

    route_master_df[
        "lane_id_from_and_to"
    ]
    .astype(str)
    .str.strip()
)

lane_master_df["lane_code"] = (

    lane_master_df["lane_name"]
    .str.upper()
    .str.replace(" ", "_")
    .str.replace("-", "_")
    .str.replace("/", "_")
    .str.replace("__", "_")
)


lane_master_df["origin_location_id"] = (
    route_master_df[
        "origin_location_id"
    ]
    .astype(str)
    .str.strip()
)
lane_master_df["origin_location_id"] = lane_master_df["origin_location_id"].apply(lambda value: str(value).zfill(4))

lane_master_df["destination_location_id"] = (

    route_master_df[
        "destination_location_id"
    ]
    .astype(str)
    .str.strip()
)

lane_master_df["destination_location_id"] = lane_master_df["destination_location_id"].apply(lambda value: str(value).zfill(4))

def normalize_transport_asset(value):
    value = str(value).upper()
    if "REEFER" in value:
        return "REEFER"
    if "VAN" in value:
        return "VAN"
    if "TRUCK" in value:
        return "TRACTOR"
    return "TRACTOR"

lane_master_df["transport_asset_type"] = (
    route_master_df[
        "transport_asset_name"
    ]
    .apply(
        normalize_transport_asset
    )
)


def safe_int(value):
    try:
        return int(float(
            str(value)
            .replace(",", "")
            .strip()
        ))
    except:
        return 0
    
    
lane_master_df["estimated_transit_hours"] = 24
lane_master_df["preferred_route_name"] = (
    route_master_df[
        "lane_type"
    ]
    .fillna("")
    .astype(str)
)

lane_master_df["distance_km"] = -1
lane_master_df["is_active"] = True



In [54]:
lane_master_df

,lane_name,lane_code,origin_location_id,destination_location_id,transport_asset_type,estimated_transit_hours,preferred_route_name,distance_km,is_active
0,Lane_From_0156_To_0155,LANE_FROM_0156_TO_0155,0156,0155,TRACTOR,24,Campus Lane,-1,True
1,Lane_From_0159_To_0156,LANE_FROM_0159_TO_0156,0159,0156,TRACTOR,24,External Input Lane,-1,True
2,Lane_From_0160_To_0156,LANE_FROM_0160_TO_0156,0160,0156,TRACTOR,24,Planning Lane,-1,True
3,Lane_From_6015_To_6006,LANE_FROM_6015_TO_6006,6015,6006,TRACTOR,24,Campus Lane,-1,True
4,Lane_From_6037_To_6010,LANE_FROM_6037_TO_6010,6037,6010,TRACTOR,24,Campus Lane,-1,True
...,...,...,...,...,...,...,...,...,...
427,Lane_From_6015_To_6008,LANE_FROM_6015_TO_6008,6015,6008,TRACTOR,24,Planning Lane,-1,True
428,Lane_From_0152_To_0151,LANE_FROM_0152_TO_0151,0152,0151,TRACTOR,24,External Input Lane,-1,True
429,Lane_From_4686_To_4641,LANE_FROM_4686_TO_4641,4686,4641,TRACTOR,24,External Input Lane,-1,True
430,Lane_From_5991_To_5998,LANE_FROM_5991_TO_5998,5991,5998,TRACTOR,24,External Input Lane,-1,True


In [55]:
lane_master_df = (
    lane_master_df
    .drop_duplicates(
        subset=[
            "origin_location_id",
            "destination_location_id",
            "transport_asset_type"
        ]
    )
    .reset_index(drop=True)
)


lane_master_df = lane_master_df[
    lane_master_df[
        "origin_location_id"
    ] != ""
]

lane_master_df = lane_master_df[

    lane_master_df[
        "destination_location_id"
    ] != ""
]

In [ ]:
# lane_master_df.to_sql(
#     name="lane_master",
#     con=engine,
#     schema="public",
#     if_exists="append",
#     index=False,
#     method="multi",
#     chunksize=1000
# )

# print(
#     "lane_master loaded successfully"
# )

lane_master loaded successfully


In [61]:
shipping_detail_df.head()

,activity2.[activity2],activity1.[activity1],orderline_id,version.[version_name],origin_location_id,estimated_delivery_date,sku_id,planned_quantity
0,0001,Lane_From_6011_To_0848,Lane_From_6011_To_0848-0001-140201-13-Apr-26-3,CurrentWorkingView,6011,13-Apr-26,140201,4320
1,0001,Lane_From_6011_To_0848,Lane_From_6011_To_0848-0001-128489-12-Apr-26-1,CurrentWorkingView,6011,12-Apr-26,128489,12960
2,0001,Lane_From_6011_To_0848,Lane_From_6011_To_0848-0001-128472-14-Apr-26-1,CurrentWorkingView,6011,14-Apr-26,128472,36720
3,0001,Lane_From_6011_To_0848,Lane_From_6011_To_0848-0001-140249-14-Apr-26-1,CurrentWorkingView,6011,14-Apr-26,140249,28080
4,0001,Lane_From_6011_To_0848,Lane_From_6011_To_0848-0001-128685-15-Apr-26-1,CurrentWorkingView,6011,15-Apr-26,128685,17280


In [74]:
shipping_detail_df.shape

(23653, 8)

In [76]:
# shipment_plans_df['estimated_delivery_date'] = 
pd.to_datetime(
            shipping_detail_df["estimated_delivery_date"],
            errors="coerce"
        ).dt.date

C:\Users\Pradeep-1093\AppData\Local\Temp\ipykernel_19140\426125636.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(


0        2026-04-13
1        2026-04-12
2        2026-04-14
3        2026-04-14
4        2026-04-15
            ...    
23648    2026-06-09
23649    2026-06-29
23650    2026-06-29
23651    2026-04-21
23652    2026-08-03
Name: estimated_delivery_date, Length: 23653, dtype: object

In [87]:
# ==========================================
# BUILD shipment_plans_df
# ==========================================

shipment_plans_df = pd.DataFrame()
shipment_plans_df['sku_id'] = (
	shipping_detail_df["sku_id"]
	.fillna(-1)
	.apply(extract_int)
	.astype("int64")
)

shipment_plans_df['planned_quantity'] = shipping_detail_df['planned_quantity'].apply(safe_int)
shipment_plans_df['planned_quantity'].fillna(0, inplace=True)
shipment_plans_df['estimated_delivery_date'] = pd.to_datetime(
            shipping_detail_df["estimated_delivery_date"],
            errors="coerce"
        ).dt.date

shipment_plans_df["shipment_id"] = shipping_detail_df[
            "orderline_id"
        ]

shipment_plans_df['actual_delivery_date'] = None
shipment_plans_df['shipped_quantity'] = 0
shipment_plans_df['destination_location_id'] = shipping_detail_df['activity1.[activity1]'].apply(lambda x: str(x)[-4:])

shipment_plans_df["origin_location_id"] =shipping_detail_df[
            "origin_location_id"
        ].astype(str).str.strip()


shipment_plans_df['priority'] = 0
shipment_plans_df['requested_transport_mode'] = 'TRUCK'

item_master_df['sku_id'] = item_master_df['sku_id'].apply(extract_int)
shipment_plans_df["weight_kg"] = (shipping_detail_df[
            "sku_id"
        ].apply(extract_int).map(
            item_master_df
            .set_index("sku_id")[
                "weight_kg"
            ]
            .to_dict()
        ).fillna(0)* shipping_detail_df[
            "planned_quantity"
        ].apply(safe_int)
)


shipment_plans_df.head()



C:\Users\Pradeep-1093\AppData\Local\Temp\ipykernel_19140\3197851298.py:14: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  shipment_plans_df['planned_quantity'].fillna(0, inplace=True)
C:\Users\Pradeep-1093\AppData\Local\Temp\ipykernel_19140\3197851298.py:15: UserWarning: Could not infer format, so each element will be parsed individua

,sku_id,planned_quantity,estimated_delivery_date,shipment_id,actual_delivery_date,shipped_quantity,destination_location_id,origin_location_id,priority,requested_transport_mode,weight_kg
0,140201,4320,2026-04-13,Lane_From_6011_To_0848-0001-140201-13-Apr-26-3,None,0,0848,6011,0,TRUCK,1.675387e+06
1,128489,12960,2026-04-12,Lane_From_6011_To_0848-0001-128489-12-Apr-26-1,None,0,0848,6011,0,TRUCK,6.924935e+06
2,128472,36720,2026-04-14,Lane_From_6011_To_0848-0001-128472-14-Apr-26-1,None,0,0848,6011,0,TRUCK,1.357456e+07
3,140249,28080,2026-04-14,Lane_From_6011_To_0848-0001-140249-14-Apr-26-1,None,0,0848,6011,0,TRUCK,1.089002e+07
4,128685,17280,2026-04-15,Lane_From_6011_To_0848-0001-128685-15-Apr-26-1,None,0,0848,6011,0,TRUCK,9.233246e+06


In [83]:
shipment_plans_df["weight_kg"].value_counts()

weight_kg
0.000000e+00    3527
9.352777e+05     128
1.332472e+06     102
1.078932e+06      89
1.210546e+05      82
                ... 
3.114959e+06       1
7.109374e+04       1
1.813329e+06       1
1.372724e+06       1
8.550708e+05       1
Name: count, Length: 8993, dtype: int64

In [88]:

# ==========================================
# LOAD INTO POSTGRESQL
# ==========================================

shipment_plans_df.to_sql(
    name="shipment_plans",
    con=engine,
    schema="public",
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000
)

print(
    "shipment_plans loaded successfully"
)

shipment_plans loaded successfully
